# 01 — Exploratory Data Analysis

Análise exploratória do dataset Inside Airbnb:
- Distribuição de preços (assimetria, outliers)
- Correlações com features numéricas
- Análise geográfica por bairro
- Análise de sazonalidade via calendário

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RAW_PATH = Path('../data/raw')
CITY = 'sao-paulo'

## 1. Carregamento dos dados

In [ ]:
listings = pd.read_csv(RAW_PATH / f'{CITY}_listings.csv.gz', compression='gzip', low_memory=False)
calendar = pd.read_csv(RAW_PATH / f'{CITY}_calendar.csv.gz', compression='gzip', parse_dates=['date'])

print(f'Listings: {listings.shape}')
print(f'Calendar: {calendar.shape}')
listings.head(3)

In [ ]:
# Overview de tipos e nulos
null_pct = (listings.isnull().sum() / len(listings) * 100).sort_values(ascending=False)
print('Colunas com > 10% de nulos:')
print(null_pct[null_pct > 10].to_string())

## 2. Distribuição de Preços

In [ ]:
# Limpeza do preço
listings['price_num'] = (
    listings['price'].astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)
df = listings[(listings['price_num'] >= 10) & (listings['price_num'] <= 5000)].copy()
df['log_price'] = np.log1p(df['price_num'])

fig, axes = plt.subplots(1, 2)

axes[0].hist(df['price_num'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de Preço (R$)')
axes[0].set_xlabel('Preço por noite (R$)')
axes[0].axvline(df['price_num'].median(), color='red', linestyle='--', label=f'Mediana: R${df["price_num"].median():.0f}')
axes[0].legend()

axes[1].hist(df['log_price'], bins=80, color='teal', edgecolor='white')
axes[1].set_title('Distribuição de log(Preço)')
axes[1].set_xlabel('log1p(Preço)')

plt.tight_layout()
plt.savefig('../data/processed/eda_price_distribution.png', bbox_inches='tight')
plt.show()

print(df['price_num'].describe())

In [ ]:
# Preço por tipo de quarto
fig, ax = plt.subplots()
order = df.groupby('room_type')['price_num'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='room_type', y='price_num', order=order, ax=ax, showfliers=False)
ax.set_title('Preço por Tipo de Acomodação')
ax.set_xlabel('')
ax.set_ylabel('Preço por noite (R$)')
plt.tight_layout()
plt.show()

## 3. Análise por Bairro

In [ ]:
# Top 20 bairros por mediana de preço
neighbourhood_stats = (
    df.groupby('neighbourhood_cleansed')['price_num']
    .agg(['median', 'count'])
    .query('count >= 30')
    .sort_values('median', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(neighbourhood_stats.index, neighbourhood_stats['median'], color='steelblue')
ax.set_xlabel('Preço mediano por noite (R$)')
ax.set_title('Top 20 Bairros — Preço Mediano')
ax.invert_yaxis()

for bar, (_, row) in zip(bars, neighbourhood_stats.iterrows()):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'n={row["count"]:.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Correlações com Preço

In [ ]:
numeric_cols = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_location',
    'calculated_host_listings_count', 'availability_365',
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

corr_data = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
corr_data['log_price'] = df['log_price']

corr = corr_data.corr()['log_price'].drop('log_price').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['green' if v > 0 else 'red' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação com log(Preço)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: accommodates vs price
fig, axes = plt.subplots(1, 2)

axes[0].scatter(df['accommodates'], df['log_price'], alpha=0.3, s=5, color='steelblue')
axes[0].set_xlabel('Capacidade (pessoas)')
axes[0].set_ylabel('log(Preço)')
axes[0].set_title('Capacidade vs Preço')

axes[1].scatter(
    pd.to_numeric(df['review_scores_rating'], errors='coerce'),
    df['log_price'], alpha=0.3, s=5, color='teal'
)
axes[1].set_xlabel('Nota média (0-5)')
axes[1].set_ylabel('log(Preço)')
axes[1].set_title('Nota vs Preço')

plt.tight_layout()
plt.show()

## 5. Sazonalidade via Calendário

In [ ]:
calendar['price_num'] = (
    calendar['price'].astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .astype(float)
)

daily = calendar.groupby('date')['price_num'].median().sort_index()
daily = daily.dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
axes[0].set_title('Preço Mediano Diário (série bruta)')
axes[0].set_ylabel('R$')

# Média por dia da semana
dow = daily.groupby(daily.index.dayofweek).mean()
dow.index = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']
axes[1].bar(dow.index, dow.values, color='teal')
axes[1].set_title('Preço Médio por Dia da Semana')
axes[1].set_ylabel('R$')

plt.tight_layout()
plt.show()

print(f'Variação fim de semana vs semana: {(dow[["Sab","Dom"]].mean() / dow[["Seg","Ter","Qua","Qui","Sex"]].mean() - 1) * 100:.1f}%')

In [ ]:
# Decomposição de sazonalidade com STL
from statsmodels.tsa.seasonal import STL

daily_clean = daily.asfreq('D').interpolate()
stl = STL(daily_clean, period=7, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], title='Observado', color='steelblue')
result.trend.plot(ax=axes[1], title='Tendência', color='orange')
result.seasonal.plot(ax=axes[2], title='Sazonalidade (semanal)', color='green')
result.resid.plot(ax=axes[3], title='Resíduo', color='gray')
plt.tight_layout()
plt.show()

## 6. Análise de Amenities

In [ ]:
import re
from collections import Counter

def parse_amenities(raw):
    if pd.isna(raw):
        return []
    return [i.lower() for i in re.findall(r'"([^"]+)"', str(raw))]

df['amenities_list'] = df['amenities'].apply(parse_amenities)

counter = Counter(a for lst in df['amenities_list'] for a in lst)
top_amenities = pd.Series(dict(counter.most_common(25)))

fig, ax = plt.subplots(figsize=(10, 7))
top_amenities.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('25 Amenities Mais Comuns')
ax.set_xlabel('Número de listings')
plt.tight_layout()
plt.show()

In [ ]:
# Impacto de amenities premium no preço
premium = ['pool', 'hot tub', 'gym', 'elevator', 'parking', 'breakfast', 'fireplace']

results = []
for amenity in premium:
    has = df['amenities_list'].apply(lambda lst: amenity in lst)
    if has.sum() > 20:
        results.append({
            'amenity': amenity,
            'price_with': df.loc[has, 'price_num'].median(),
            'price_without': df.loc[~has, 'price_num'].median(),
            'count': has.sum(),
        })

amenity_impact = pd.DataFrame(results).set_index('amenity')
amenity_impact['premium_pct'] = (amenity_impact['price_with'] / amenity_impact['price_without'] - 1) * 100
amenity_impact = amenity_impact.sort_values('premium_pct', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['green' if v > 0 else 'red' for v in amenity_impact['premium_pct']]
amenity_impact['premium_pct'].plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Premium de Preço por Amenity (%)')
ax.set_xlabel('% acima da mediana sem a amenity')
plt.tight_layout()
plt.show()

print(amenity_impact[['price_with', 'price_without', 'premium_pct', 'count']].to_string())

## 7. Conclusões do EDA

- **Preço é assimétrico**: usar `log(price)` como target é essencial
- **Sazonalidade semanal existe**: fins de semana têm preço ~X% maior → Holt-Winters captura isso
- **Bairro é forte preditor**: grande variação entre bairros → Target Encoding
- **Capacidade é a feature mais correlacionada**: accommodates, bedrooms, beds
- **Amenities premium**: pool, hot tub e gym são os maiores diferenciais de preço